# Sanjaya Engine on Google Colab

Turn your own photos and videos into **measured 3D maps, floor plans, mental maps and object lists**, using a free Colab GPU and exactly the same code as the public [Sanjaya Engine Space](https://huggingface.co/spaces/netha01/sanjaya-engine).

**What this notebook is for:** processing your recordings, checking results, and producing the sample missions shown on the website and in the Space. Results are saved to your Google Drive.

**What it is not for:** Colab does not allow hosting public web apps, so the public demo stays on Hugging Face. Everything here runs inside notebook cells.

### Steps
1. **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.
2. Put your recordings in **Google Drive → `sanjaya/input/`**:
   - one **video** file = one mission (`corridor-walk.mp4`)
   - one **folder of photos** = one mission (`office-room/000.jpg, 001.jpg, …`, taken while walking, in order, each overlapping the last)
3. Results appear in **`sanjaya/output/<mission>/`**.

In [ ]:
# 1. Check the GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no GPU found")
assert torch.cuda.is_available(), "No GPU. Use Runtime -> Change runtime type -> T4 GPU, then run again."

In [ ]:
# 2. Get the engine (same code as the public Space)
import os
ENGINE_REPO = "https://huggingface.co/spaces/netha01/sanjaya-engine"
if os.path.isdir("/content/sanjaya-engine/.git"):
    !git -C /content/sanjaya-engine pull -q
else:
    !git clone -q --depth 1 {ENGINE_REPO} /content/sanjaya-engine
%cd /content/sanjaya-engine
!ls sanjaya

In [ ]:
# 3. Install (about 2 minutes)
!pip -q install -r requirements.txt "gradio==5.50.0" plotly

In [ ]:
# 4. Connect Google Drive and create the folders
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/sanjaya"
INPUT_DIR, OUTPUT_DIR, SAMPLES_DIR = f"{BASE}/input", f"{BASE}/output", f"{BASE}/samples"
for d in (INPUT_DIR, OUTPUT_DIR, SAMPLES_DIR):
    os.makedirs(d, exist_ok=True)
print("Put videos or photo folders in:", INPUT_DIR)

**Optional:** upload photos straight from your computer instead of Drive. Run the next cell, choose the photos of ONE walk, and they become one mission.

In [ ]:
# 4b. (optional) Upload one mission's photos or video from this computer
from google.colab import files
MISSION_NAME = "my-first-walk"   # change for each upload
uploaded = files.upload()
if uploaded:
    names = list(uploaded)
    if len(names) == 1 and names[0].lower().endswith((".mp4", ".mov", ".m4v", ".webm")):
        os.replace(names[0], f"{INPUT_DIR}/{MISSION_NAME}{os.path.splitext(names[0])[1]}")
    else:
        os.makedirs(f"{INPUT_DIR}/{MISSION_NAME}", exist_ok=True)
        for n in names:
            os.replace(n, f"{INPUT_DIR}/{MISSION_NAME}/{n}")
    print("Saved", len(names), "file(s) as mission", MISSION_NAME)

In [ ]:
# 5. Load the models (first time: about 3 minutes, 5 GB download)
import app                                   # loads LingBot-Map, Depth Anything V2 Metric and OWLv2
from sanjaya import notebook as NB
from IPython.display import display, Markdown, Image

In [ ]:
# 6. See which missions are waiting
missions = NB.find_missions(INPUT_DIR)
for m in missions:
    kind = "video" if m["video"] else f"{len(m['photos'])} photos"
    done = os.path.isfile(f"{OUTPUT_DIR}/{NB.slugify(m['name'])}/sanjaya_mission.zip")
    print(f"{'done   ' if done else 'waiting'}  {m['name']}  ({kind})")

### 7. Map them
Settings you may want to change:
- `SCENE`: `"Indoor"` or `"Outdoor"` (picks the right real-world scale model)
- `THINGS`: what to look for, in plain words. Leave `None` for the default list (doors, stairs, people, exits…)
- `REDO`: `True` to process missions again

In [ ]:
# 7. Map every waiting mission and show the results
SCENE, THINGS, REDO = "Indoor", None, False
results = {}
for m in missions:
    slug = NB.slugify(m["name"])
    if not REDO and os.path.isfile(f"{OUTPUT_DIR}/{slug}/sanjaya_mission.zip"):
        continue
    display(Markdown(f"## {m['name']}"))
    try:
        r = NB.run_mission(app, m, OUTPUT_DIR, scene=SCENE, things=THINGS)
    except Exception as e:
        display(Markdown(f"**Failed:** {e}"))
        continue
    results[m["name"]] = r
    display(Markdown(r["status"]))
    NB.preview_3d(r["state"]).show()
    for img in (r["plan"], r["mental_map"]):
        if img:
            display(Image(img, width=560))
    for path, caption in r["gallery"][:8]:
        display(Markdown(f"*{caption}*"))
        display(Image(path, width=200))
print("Saved to", OUTPUT_DIR)

### 8. Turn the best missions into samples for the Space and the website
Give each mission a title and a one-line description. Missions not listed are skipped.

In [ ]:
# 8. Make samples (re-encodes videos to under 10 MB, no audio)
SAMPLES = {
    # "mission-name": ("Title shown on the site", "One-line description"),
    # "corridor-walk": ("Corridor and back", "A 30-second phone walk down a college corridor and back."),
}
by_name = {m["name"]: m for m in missions}
for name, (title, desc) in SAMPLES.items():
    m = by_name[name]
    out = NB.make_sample(f"{OUTPUT_DIR}/{NB.slugify(name)}", m["video"] or m["photos"], title, desc, SAMPLES_DIR)
    print("sample ready:", out)

In [ ]:
# 9. (optional) Publish the samples to your Space
# Add a Hugging Face token with WRITE access as a Colab secret named HF_TOKEN
# (key icon in the left sidebar), then run this cell. The Space restarts with the new samples.
from google.colab import userdata
from huggingface_hub import HfApi
HfApi(token=userdata.get("HF_TOKEN")).upload_folder(
    repo_id="netha01/sanjaya-engine", repo_type="space",
    folder_path=SAMPLES_DIR, path_in_repo="samples",
    commit_message="Add sample missions from Colab")
print("Published. Open the Space's 'Sample missions' tab in a few minutes.")

### Tips for recordings that map well
- Walk slowly; turn while moving, never spin in place.
- Landscape, 1080p, good light; tilt slightly down so the floor is visible.
- 20–30 s of video, or 40–100 photos taken every half step.
- End where you started to show "returned to start".
- Only film places you have permission to film.